In [2]:
import pandas as pd
import numpy as np
import re

raw_df = pd.read_csv("../data/raw/raw_arabam.csv")

print(f"Total row: {len(raw_df)}, \nTotal column: {len(raw_df.columns)}")


Total row: 1000, 
Total column: 30


In [3]:
print(list(raw_df.columns))

['url', 'title', 'price', 'location', 'i̇lan no', 'i̇lan tarihi', 'marka', 'seri', 'model', 'yıl', 'kilometre', 'vites tipi', 'yakıt tipi', 'kasa tipi', 'renk', 'motor hacmi', 'motor gücü', 'çekiş', 'araç durumu', 'ort. yakıt tüketimi', 'yakıt deposu', 'ağır hasarlı', 'boya-değişen', 'takasa uygun', 'kimden', 'motor gücü (kw)', 'batarya voltajı (v)', 'menzil', 'şarj süresi', 'pil kapasitesi (kwh)']


## Numeric

### Price

In [4]:
print("Unique Price Count:", raw_df["price"].nunique())

print("\nSample Unique Prices:")
print(raw_df["price"].dropna().unique()[:15])

Unique Price Count: 575

Sample Unique Prices:
['1.135.000 TL' '660.000 TL' '1.380.000 TL' '445.000 TL' '230.000 TL'
 '590.000 TL' '825.000 TL' '210.000 TL' '395.000 TL' '780.000 TL'
 '548.800 TL' '797.500 TL' '498.500 TL' '1.015.000 TL' '875.000 TL']


In [5]:
anomalies = raw_df[~raw_df["price"].astype(str).str.contains(r'TL', case=False, na=False)]

print("Non-TL or Anomalous Price Count:", len(anomalies))
if len(anomalies) > 0:
    print("Anomalous Prices Found:")
    print(anomalies["price"].unique())
else:
    print("All price entries contain 'TL'.")

Non-TL or Anomalous Price Count: 0
All price entries contain 'TL'.


In [6]:
def clean_price(val) -> float:
    """
    Cleans Turkish Lira price strings into float numbers.
    Example: '1.135.000 TL' -> 1135000.0
    """

    if pd.isna(val):
        return np.nan

    cleaned_str = str(val).replace(".", "").replace("TL", "").replace("tl", "").strip()

    try:
        return float(cleaned_str)
    except ValueError:
        return np.nan

In [7]:
sample_input = "1.135.000 TL"
print(f"Test Dönüşümü: '{sample_input}' -> {clean_price(sample_input)} (Tip: {type(clean_price(sample_input))})")

cleaned_prices = raw_df["price"].apply(clean_price)
print("\nTemizlenmiş İlk 5 Fiyat:")
print(cleaned_prices.head())

print("\nDönüştürülemeyen (NaN) Fiyat Sayısı:", cleaned_prices.isna().sum())

Test Dönüşümü: '1.135.000 TL' -> 1135000.0 (Tip: <class 'float'>)

Temizlenmiş İlk 5 Fiyat:
0    1135000.0
1     660000.0
2    1380000.0
3     445000.0
4     230000.0
Name: price, dtype: float64

Dönüştürülemeyen (NaN) Fiyat Sayısı: 67


In [8]:
def clean_price(val) -> float:
    """
    Cleans Turkish Lira price strings into float numbers.
    Example: '1.135.000 TL' -> 1135000.0
    """

    if pd.isna(val):
        return np.nan

    cleaned_str = str(val).replace(".", "").replace("TL", "").replace("tl", "").strip()

    try:
        return float(cleaned_str)
    except ValueError:
        return np.nan

sample_input = "1.135.000 TL"
print(f"Test Conversion: '{sample_input}' -> {clean_price(sample_input)} (Type: {type(clean_price(sample_input))})")

cleaned_prices = raw_df["price"].apply(clean_price)
print("\nCleaned Top 5 Prices:")
print(cleaned_prices.head())

print("\nUnconverted (NaN) Price Count:", cleaned_prices.isna().sum())

Test Conversion: '1.135.000 TL' -> 1135000.0 (Type: <class 'float'>)

Cleaned Top 5 Prices:
0    1135000.0
1     660000.0
2    1380000.0
3     445000.0
4     230000.0
Name: price, dtype: float64

Unconverted (NaN) Price Count: 67


In [9]:
failed_price_mask = cleaned_prices.isna()
failed_original_prices = raw_df.loc[failed_price_mask, "price"]

print("Failed Price Inspection (Sample Unique Values):")
print(failed_original_prices.unique()[:20])

print("\nAlready Missing (NaN) in Raw Data:", raw_df["price"].isna().sum())

Failed Price Inspection (Sample Unique Values):
['895.000 TL880.000 TL' '748.000 TL718.000 TL' '1.365.000 TL1.345.000 TL'
 '1.155.000 TL1.139.000 TL' '1.200.000 TL1.115.000 TL'
 '1.882.000 TL1.862.000 TL' '1.399.000 TL1.349.000 TL'
 '845.000 TL825.000 TL' '1.195.000 TL1.175.000 TL' '905.000 TL880.000 TL'
 '1.377.000 TL1.347.000 TL' '385.000 TL370.000 TL' '929.000 TL909.000 TL'
 '745.000 TL715.000 TL' '745.000 TL730.000 TL' '825.000 TL795.000 TL'
 '1.345.000 TL1.325.000 TL' '1.155.000 TL1.125.000 TL'
 '805.000 TL795.000 TL' '689.000 TL675.000 TL']

Already Missing (NaN) in Raw Data: 0


In [10]:
def clean_price(val) -> float:
    """
    Cleans Turkish Lira price strings into float numbers.
    If discounted (e.g. '895.000 TL880.000 TL'), takes the final current price.
    """
    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip()

    price_matches = re.findall(r'(\d+(?:\.\d+)*)', val_str)

    if not price_matches:
        return np.nan

    final_price_str = price_matches[-1].replace(".", "")

    try:
        return float(final_price_str)
    except ValueError:
        return np.nan

discounted_sample = "895.000 TL880.000 TL"
normal_sample = "1.135.000 TL"

print(f"Discounted Test: '{discounted_sample}' -> {clean_price(discounted_sample)}")
print(f"Normal Test:     '{normal_sample}' -> {clean_price(normal_sample)}")

cleaned_prices = raw_df["price"].apply(clean_price)

print("\nUnconverted (NaN) Price Count:", cleaned_prices.isna().sum())

Discounted Test: '895.000 TL880.000 TL' -> 880000.0
Normal Test:     '1.135.000 TL' -> 1135000.0

Unconverted (NaN) Price Count: 0


### KM

In [11]:
km_col = "kilometre"

print("Total Rows:", len(raw_df))
print(f"Unique Count in '{km_col}':", raw_df[km_col].nunique())
print(f"Missing (NaN) Count in Raw Data:", raw_df[km_col].isna().sum())

print(f"\nSample Unique Values in '{km_col}':")
print(raw_df[km_col].dropna().unique()[:15])

Total Rows: 1000
Unique Count in 'kilometre': 447
Missing (NaN) Count in Raw Data: 0

Sample Unique Values in 'kilometre':
['98.800 KM' '283.000 KM' '172.000 KM' '129.000 KM' '200.000 KM'
 '199.000 KM' '209.000 KM' '366.000 KM' '255.000 KM' '170.000 KM'
 '209.001 KM' '138.000 KM' '398.800 KM' '203.450 KM' '325.000 KM']


In [12]:
km_anomalies = raw_df[~raw_df["kilometre"].astype(str).str.contains(r'km', case=False, na=False)]

print("Non-KM Anomalous Count:", len(km_anomalies))
if len(km_anomalies) > 0:
    print("Anomalous KM Values:")
    print(km_anomalies["kilometre"].unique())
else:
    print("All kilometre entries contain 'KM'.")

Non-KM Anomalous Count: 0
All kilometre entries contain 'KM'.


In [13]:
def clean_km(val) -> float:
    """
    Cleans kilometre strings into float numbers.
    Example: '98.800 KM' -> 98800.0
    """
    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip()

    cleaned_str = re.sub(r'[^0-9]', '', val_str)
    
    if not cleaned_str:
        return np.nan

    try:
        return float(cleaned_str)
    except ValueError:
        return np.nan

sample_km = "98.800 KM"
sample_zero_km = "0 KM"

print(f"Test KM:      '{sample_km}' -> {clean_km(sample_km)} (Type: {type(clean_km(sample_km))})")
print(f"Test Zero KM: '{sample_zero_km}' -> {clean_km(sample_zero_km)}")

cleaned_kms = raw_df["kilometre"].apply(clean_km)

print("\nCleaned Top 5 Kilometres:")
print(cleaned_kms.head())

print("\nUnconverted (NaN) Kilometre Count:", cleaned_kms.isna().sum())

Test KM:      '98.800 KM' -> 98800.0 (Type: <class 'float'>)
Test Zero KM: '0 KM' -> 0.0

Cleaned Top 5 Kilometres:
0     98800.0
1    283000.0
2    172000.0
3    129000.0
4    200000.0
Name: kilometre, dtype: float64

Unconverted (NaN) Kilometre Count: 0


### Engine Size

In [14]:
engine_col = "motor hacmi"

print("Total Rows:", len(raw_df))
print(f"Unique Count in '{engine_col}':", raw_df[engine_col].nunique())
print(f"Missing (NaN) Count in Raw Data:", raw_df[engine_col].isna().sum())

print(f"\nSample Unique Values in '{engine_col}':")
print(raw_df[engine_col].dropna().unique()[:20])

Total Rows: 1000
Unique Count in 'motor hacmi': 81
Missing (NaN) Count in Raw Data: 4

Sample Unique Values in 'motor hacmi':
['1496 CC' '1422 CC' '1798 CC' '1399 CC' '1581 CC' '1461 CC' '1329 CC'
 '1910 CC' '1149 CC' '1396 CC' '1360 CC' '1201 - 1400 cm3' '1364 CC'
 '1598 CC' '1560 CC' '1401 - 1600 cm3' '1395 CC' '1596 CC' '1368 CC'
 '1968 CC']


In [15]:
engine_series = raw_df["motor hacmi"].dropna().astype(str)

range_formats = engine_series[engine_series.str.contains("-")]

print("Range Format Count:", len(range_formats))
print("Sample Range Formats:")
print(range_formats.unique())

other_formats = engine_series[~engine_series.str.contains(r'CC|cm3', case=False, regex=True)]
print("\nOther Formats Count:", len(other_formats))
if len(other_formats) > 0:
    print(other_formats.unique())

Range Format Count: 259
Sample Range Formats:
['1201 - 1400 cm3' '1401 - 1600 cm3' '1801 - 2000 cm3' '2001 - 2500 cm3'
 '1601 - 1800 cm3' '2501 - 3000 cm3' '4501 - 5000 cm3' '-']

Other Formats Count: 2
['-']


In [16]:
def clean_engine_size(val) -> float:
    """
    Cleans engine displacement strings into float (cc).
    Handles exact values ('1598 CC') and ranges ('1201 - 1400 cm3' -> midpoint).
    """

    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip()

    numbers = [float(n) for n in re.findall(r'\d+', val_str)]

    if not numbers:
        return np.nan

    if len(numbers) >= 2:
        return (numbers[0] + numbers[1]) / 2.0

    return numbers[0]

sample_exact = "1598 CC"
sample_range = "1201 - 1400 cm3"
sample_dash = "-"

print(f"Exact Test: '{sample_exact}' -> {clean_engine_size(sample_exact)}")
print(f"Range Test: '{sample_range}' -> {clean_engine_size(sample_range)}")
print(f"Dash Test:  '{sample_dash}' -> {clean_engine_size(sample_dash)}")

cleaned_engines = raw_df["motor hacmi"].apply(clean_engine_size)

print("\nCleaned Top 5 Engine Sizes:")
print(cleaned_engines.head())

print("\nUnconverted or Missing Engine Size Count:", cleaned_engines.isna().sum())

Exact Test: '1598 CC' -> 1598.0
Range Test: '1201 - 1400 cm3' -> 1300.5
Dash Test:  '-' -> nan

Cleaned Top 5 Engine Sizes:
0    1496.0
1    1422.0
2    1798.0
3    1399.0
4    1581.0
Name: motor hacmi, dtype: float64

Unconverted or Missing Engine Size Count: 6


### Engine Power

In [17]:
power_col = "motor gücü"

print("Total Rows:", len(raw_df))
print(f"Unique Count in '{power_col}':", raw_df[power_col].nunique())
print(f"Missing (NaN) Count in Raw Data:", raw_df[power_col].isna().sum())

print(f"\nSample Unique Values in '{power_col}':")
print(raw_df[power_col].dropna().unique()[:20])

Total Rows: 1000
Unique Count in 'motor gücü': 89
Missing (NaN) Count in Raw Data: 0

Sample Unique Values in 'motor gücü':
['111 HP' '90 HP' '122 HP' '94 HP' '83 HP' '65 HP' '99 HP' '116 HP'
 '75 HP' '108 HP' '71 HP' '76 - 100 HP' '105 HP' '95 HP' '101 - 125 HP'
 '150 HP' '100 HP' '115 HP' '180 HP' '77 HP']


In [18]:
power_series = raw_df["motor gücü"].dropna().astype(str)

range_power = power_series[power_series.str.contains("-")]

print("Range Format Count in 'motor gücü':", len(range_power))
print("Sample Range Formats:")
print(range_power.unique())

other_power = power_series[~power_series.str.contains(r'HP', case=False, regex=True)]
print("\nNon-HP Format Count:", len(other_power))
if len(other_power) > 0:
    print(other_power.unique())

Range Format Count in 'motor gücü': 299
Sample Range Formats:
['76 - 100 HP' '101 - 125 HP' '126 - 150 HP' '301 - 325 HP' '151 - 175 HP'
 '51 - 75 HP' '176 - 200 HP' '201 - 225 HP' '226 - 250 HP' '251 - 275 HP'
 '526 - 550 HP']

Non-HP Format Count: 0


In [19]:
def clean_engine_power(val) -> float:

    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip()

    numbers = [float(n) for n in re.findall(r'\d+', val_str)]
    
    if not numbers:
        return np.nan

    if len(numbers) >= 2:
        return (numbers[0] + numbers[1]) / 2.0

    return numbers[0]

sample_exact_power = "111 HP"
sample_range_power = "76 - 100 HP"

print(f"Exact Test: '{sample_exact_power}' -> {clean_engine_power(sample_exact_power)}")
print(f"Range Test: '{sample_range_power}' -> {clean_engine_power(sample_range_power)}")

cleaned_powers = raw_df["motor gücü"].apply(clean_engine_power)

print("\nCleaned Top 5 Engine Powers:")
print(cleaned_powers.head())

print("\nUnconverted or Missing Power Count:", cleaned_powers.isna().sum())

Exact Test: '111 HP' -> 111.0
Range Test: '76 - 100 HP' -> 88.0

Cleaned Top 5 Engine Powers:
0    111.0
1     90.0
2    122.0
3     94.0
4     83.0
Name: motor gücü, dtype: float64

Unconverted or Missing Power Count: 0


### Year

In [20]:
year_col = "yıl"

print("Total Rows:", len(raw_df))
print(f"Unique Count in '{year_col}':", raw_df[year_col].nunique())
print(f"Missing (NaN) Count in Raw Data:", raw_df[year_col].isna().sum())

print(f"\nSample Unique Values in '{year_col}':")
print(raw_df[year_col].dropna().unique()[:15])
print(f"\nCurrent Data Type: {raw_df[year_col].dtype}")

Total Rows: 1000
Unique Count in 'yıl': 40
Missing (NaN) Count in Raw Data: 0

Sample Unique Values in 'yıl':
[2017 2014 2021 2006 1994 2010 2015 2002 2008 2012 2022 2007 2013 2011
 2004]

Current Data Type: int64


In [21]:
min_year = raw_df["yıl"].min()
max_year = raw_df["yıl"].max()

print("Minimum Model Year:", min_year)
print("Maximum Model Year:", max_year)

year_anomalies = raw_df[(raw_df["yıl"] < 1970) | (raw_df["yıl"] > 2026)]
print("Anomalous Year Count:", len(year_anomalies))

Minimum Model Year: 1984
Maximum Model Year: 2026
Anomalous Year Count: 0


In [22]:
def clean_year(val) -> float:

    if pd.isna(val):
        return np.nan

    try:
        year_num = float(val)
        if 1970 <= year_num <= 2026:
            return year_num
        return np.nan
    except (ValueError, TypeError):
        return np.nan

cleaned_years = raw_df["yıl"].apply(clean_year)
print("Unconverted or Out-of-Range Year Count:", cleaned_years.isna().sum())

Unconverted or Out-of-Range Year Count: 0


### Fuel Consumption

In [23]:
fuel_cons_col = "ort. yakıt tüketimi"

print("Total Rows:", len(raw_df))
print(f"Unique Count in '{fuel_cons_col}':", raw_df[fuel_cons_col].nunique())
print(f"Missing (NaN) Count in Raw Data:", raw_df[fuel_cons_col].isna().sum())

print(f"\nSample Unique Values in '{fuel_cons_col}':")
print(raw_df[fuel_cons_col].dropna().unique()[:20])

Total Rows: 1000
Unique Count in 'ort. yakıt tüketimi': 52
Missing (NaN) Count in Raw Data: 324

Sample Unique Values in 'ort. yakıt tüketimi':
['4,8 lt' '3,5 lt' '7,6 lt' '4,5 lt' '5,8 lt' '5,3 lt' '5,9 lt' '6,3 lt'
 '4,4 lt' '4,9 lt' '4,3 lt' '5,5 lt' '6,7 lt' '6,9 lt' '7,7 lt' '6,2 lt'
 '6,4 lt' '6 lt' '7,5 lt' '4,2 lt']


In [24]:
fuel_series = raw_df["ort. yakıt tüketimi"].dropna().astype(str)

non_lt = fuel_series[~fuel_series.str.contains(r'lt', case=False)]
print("Non-'lt' Format Count:", len(non_lt))
if len(non_lt) > 0:
    print("Non-'lt' Values:", non_lt.unique())

anomalies_fuel = fuel_series[fuel_series.str.contains(r'[^0-9,\.\sltLT]')]
print("Anomalous Fuel Consumption Count:", len(anomalies_fuel))
if len(anomalies_fuel) > 0:
    print("Anomalies Found:", anomalies_fuel.unique())

Non-'lt' Format Count: 0
Anomalous Fuel Consumption Count: 0


In [25]:
def clean_fuel_consumption(val) -> float:

    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip()

    cleaned_str = val_str.replace(",", ".").replace("lt", "").replace("LT", "").strip()

    try:
        cons_val = float(cleaned_str)

        if 0 < cons_val <= 30.0:
            return cons_val
        return np.nan
    except ValueError:
        return np.nan

sample_decimal = "4,8 lt"
sample_integer = "6 lt"

print(f"Decimal Test: '{sample_decimal}' -> {clean_fuel_consumption(sample_decimal)}")
print(f"Integer Test: '{sample_integer}' -> {clean_fuel_consumption(sample_integer)}")

cleaned_fuel = raw_df["ort. yakıt tüketimi"].apply(clean_fuel_consumption)

print("\nCleaned Top 5 Fuel Consumption Values:")
print(cleaned_fuel.head())

print("\nTotal Missing (NaN) Fuel Count:", cleaned_fuel.isna().sum())

Decimal Test: '4,8 lt' -> 4.8
Integer Test: '6 lt' -> 6.0

Cleaned Top 5 Fuel Consumption Values:
0    4.8
1    3.5
2    3.5
3    7.6
4    NaN
Name: ort. yakıt tüketimi, dtype: float64

Total Missing (NaN) Fuel Count: 324


### Damage Parsing

In [26]:
damage_col = "boya-değişen"

print("Total Rows: ", len(raw_df))
print(f"Unique Count in '{damage_col}': ", raw_df[damage_col].nunique())
print("Missing (NaN) Count in Raw Data: ", raw_df[damage_col].isna().sum())

print(f"\nTop 15 Most Frequent Values in '{damage_col}': ")
print(raw_df[damage_col].value_counts().head(15))

Total Rows:  1000
Unique Count in 'boya-değişen':  117
Missing (NaN) Count in Raw Data:  0

Top 15 Most Frequent Values in 'boya-değişen': 
boya-değişen
Tamamı orjinal              238
Belirtilmemiş               111
1 boyalı                     42
1 lokal boyalı               40
3 boyalı                     36
1 boyalı, 1 lokal boyalı     34
2 boyalı                     27
1 değişen                    24
2 lokal boyalı               21
1 değişen, 3 boyalı          19
4 boyalı                     17
3 boyalı, 1 lokal boyalı     17
2 boyalı, 1 lokal boyalı     17
1 değişen, 1 boyalı          16
3 lokal boyalı               16
Name: count, dtype: int64


In [27]:
def parse_damage_status(val) -> dict:
    """
    Parses 'boya-değişen' text into 4 structured numerical features:
    - painted_count 
    - local_painted_count
    - changed_count
    - is_original (1 or 0)

    """

    if pd.isna(val) or str(val).strip() == "Belirtilmemiş":
        return {
            "painted_count" : np.nan,
            "local_painted_count" : np.nan,
            "changed_count" : np.nan,
            "is_original" : 0 
        }

    val_str = str(val).strip().lower()

    if "tamamı orjinal" in val_str or "tamamı orijinal" in val_str:
        return {
            "painted_count" : 0,
            "local_painted_count" : 0,
            "changed_count" : 0,
            "is_original" : 1 
        }

    local_match = re.search(r"(\d+)\s*lokal\s*boyalı", val_str)
    local_cnt = float(local_match.group(1)) if local_match else 0

    val_without_local = re.sub(r"\d+\s*lokal\s*boyalı", "", val_str)
    painted_match = re.search(r"(\d+)\s*boyalı", val_without_local)
    painted_cnt = float(painted_match.group(1)) if painted_match else 0

    changed_match = re.search(r"(\d+)\s*değişen", val_str)
    changed_cnt = float(changed_match.group(1)) if changed_match else 0

    return {
        "painted_count" : painted_cnt,
        "local_painted_count" : local_cnt,
        "changed_count" : changed_cnt,
        "is_original" : 0
    }

sample_cases = [
    "Tamamı orjinal",
    "Belirtilmemiş",
    "1 değişen, 3 boyalı",
    "2 boyalı, 1 lokal boyalı",
    "1 değişen"
]

print("Damage Parsing Test: ")
for case in sample_cases:
    print(f" '{case}' -> {parse_damage_status(case)}")

damage_features_df = raw_df["boya-değişen"].apply(parse_damage_status).apply(pd.Series)

print("\nDamage Feature Columns (First 5 Rows): ")
print(damage_features_df.head())

print("\nDamage Summary Distributions: ")
print(damage_features_df.describe())

Damage Parsing Test: 
 'Tamamı orjinal' -> {'painted_count': 0, 'local_painted_count': 0, 'changed_count': 0, 'is_original': 1}
 'Belirtilmemiş' -> {'painted_count': nan, 'local_painted_count': nan, 'changed_count': nan, 'is_original': 0}
 '1 değişen, 3 boyalı' -> {'painted_count': 3.0, 'local_painted_count': 0, 'changed_count': 1.0, 'is_original': 0}
 '2 boyalı, 1 lokal boyalı' -> {'painted_count': 2.0, 'local_painted_count': 1.0, 'changed_count': 0, 'is_original': 0}
 '1 değişen' -> {'painted_count': 0, 'local_painted_count': 0, 'changed_count': 1.0, 'is_original': 0}

Damage Feature Columns (First 5 Rows): 
   painted_count  local_painted_count  changed_count  is_original
0            0.0                  0.0            0.0          1.0
1            7.0                  0.0            2.0          0.0
2            0.0                  1.0            0.0          0.0
3           10.0                  0.0            0.0          0.0
4            0.0                  0.0            0.0

## Categoric

In [28]:
cat_cols = ["yakıt tipi", "vites tipi", "kasa tipi", "çekiş", "araç durumu"]

for col in cat_cols:
    print(f"\n--- {col.upper()} (Unique Count: {raw_df[col].nunique()}) ---")
    print(raw_df[col].value_counts(dropna=False))


--- YAKIT TIPI (Unique Count: 5) ---
yakıt tipi
Dizel           422
Benzin          375
LPG & benzin    165
Hibrit           32
Elektrik          6
Name: count, dtype: int64

--- VITES TIPI (Unique Count: 3) ---
vites tipi
Düz              449
Otomatik         379
Yarı Otomatik    172
Name: count, dtype: int64

--- KASA TIPI (Unique Count: 9) ---
kasa tipi
Sedan            552
Hatchback/5      378
Station wagon     19
Hatchback/3       17
Coupe             15
MPV               13
-                  3
SUV                2
Cabrio             1
Name: count, dtype: int64

--- ÇEKIŞ (Unique Count: 4) ---
çekiş
Önden Çekiş      873
Arkadan İtiş     108
4WD (Sürekli)     18
-                  1
Name: count, dtype: int64

--- ARAÇ DURUMU (Unique Count: 1) ---
araç durumu
İkinci El    1000
Name: count, dtype: int64


In [29]:
def clean_category(val) -> str:
    if pd.isna(val):
        return np.nan

    val_str = str(val).strip()

    if val_str in ["-", "", "Belirtilmemiş"]:
        return np.nan

    return val_str.title()

print("Category Test: ", clean_category("   dizel   "))

Category Test:  Dizel


## Sanity Check

In [31]:
processed_df = pd.read_csv("../data/processed/cleaned_arabam.csv")

print("Cleaned Dataset Shape: ", processed_df.shape)
print("\n--- Column Data Types ---")
print(processed_df.dtypes)

print("\n--- Missing Value Summary (Per Column) ---")
print(processed_df.isna().sum())

print("\n--- Cleaned Data Sample (First 3 Rows) ---")
processed_df.head(3)

Cleaned Dataset Shape:  (1000, 18)

--- Column Data Types ---
price                     float64
brand                      object
series                     object
model                      object
year                      float64
kilometer                 float64
transmission               object
fuel_type                  object
body_type                  object
color                      object
engine_displacement_cc    float64
engine_power_hp           float64
drivetrain                 object
avg_fuel_consumption      float64
painted_count             float64
local_painted_count       float64
changed_count             float64
is_original               float64
dtype: object

--- Missing Value Summary (Per Column) ---
price                       0
brand                       0
series                      0
model                       1
year                        0
kilometer                   0
transmission                0
fuel_type                   0
body_type                   

,price,brand,series,model,year,kilometer,transmission,fuel_type,body_type,color,engine_displacement_cc,engine_power_hp,drivetrain,avg_fuel_consumption,painted_count,local_painted_count,changed_count,is_original
0,1135000.0,Toyota,Yaris,1.5 Fun Special,2017.0,98800.0,Otomatik,Benzin,Hatchback/5,Beyaz,1496.0,111.0,Önden Çekiş,4.8,0.0,0.0,0.0,1.0
1,660000.0,Volkswagen,Polo,1.4 Tdi Comfortline,2014.0,283000.0,Yarı Otomatik,Dizel,Hatchback/5,Siyah,1422.0,90.0,Önden Çekiş,3.5,7.0,0.0,2.0,0.0
2,1380000.0,Toyota,Corolla,1.8 Hybrid Dream,2021.0,172000.0,Otomatik,Hibrit,Sedan,Beyaz,1798.0,122.0,Önden Çekiş,3.5,0.0,1.0,0.0,0.0
